# Enforce Knowledge — Varying N_SAMPLES and N_LOCAL together

Two-phase experiment in a **single chat session** per model:
- **Phase 1:** Model info + columns + train sample + real vs predicted → analysis (no XAI)
- **Phase 2:** Same chat, SHAP global + SHAP local + LIME local injected → revised analysis

Configurations:
| Config | N_SAMPLES | N_LOCAL |
|--------|-----------|--------|
| 1      | 10        | 10     |
| 2      | 20        | 15     |
| 3      | 40        | 25     |

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import shap
import lime
import warnings
from lime import lime_tabular
from openai import OpenAI
import httpx
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
REQUEST_TIMEOUT = 900.0

models = [
    {"name": "glm-4.7-flash:latest",  "size": "medium"},
    {"name": "qwen3:14b",              "size": "medium"},
    {"name": "gpt-oss:20b",            "size": "medium"},
    {"name": "qwen3:30b",              "size": "large"},
]

# Paired configs: (N_SAMPLES, N_LOCAL)
CONFIGS = [
    (10, 10),
    (20, 15),
    (40, 25),
]

N_LOCAL_MAX = max(n_local for _, n_local in CONFIGS)

# Leitura e Pre-processamento dos Dados

In [ ]:
df = pd.read_csv("../Network_logs.csv")

In [ ]:
networkData = df.copy()
networkData.drop(['Source_IP', 'Destination_IP', 'Intrusion'], axis=1, inplace=True)
networkData.head(5)

In [ ]:
categorical_cols = ['Request_Type', 'Protocol', 'User_Agent', 'Status', 'Port']
for col in categorical_cols:
    networkData[col] = networkData[col].astype('category')

for col in categorical_cols:
    print(f"{col} categories: {networkData[col].cat.categories.tolist()}")

for col in categorical_cols:
    networkData[col] = networkData[col].cat.codes

In [ ]:
target_encoder = LabelEncoder()
networkData['Scan_Type_Label'] = target_encoder.fit_transform(networkData['Scan_Type'])

label_mapping = dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))
print("Label Mapping:", label_mapping)

networkData.drop(['Scan_Type'], axis=1, inplace=True)
networkData.head(5)

In [ ]:
scaler = StandardScaler()
networkData['Payload_Size'] = scaler.fit_transform(networkData[['Payload_Size']])

X = networkData.drop(['Scan_Type_Label'], axis=1)
y = networkData['Scan_Type_Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
smote = SMOTE()
X_train, y_train = smote.fit_resample(X_train, y_train)
y_train = pd.Series(y_train.values.ravel(), name='Scan_Type_Label')

print('SMOTE aplicado com sucesso.\n')
print('Nova distribuição:\n')
print(y_train.value_counts())

# Treinamento do Modelo

In [ ]:
rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Acurácia: {acc:.4f}")
print(classification_report(y_test, y_pred))

# Explicabilidade SHAP + LIME (compute once for N_LOCAL_MAX)

In [ ]:
feature_names = list(X.columns)
class_names = list(target_encoder.classes_)

np.random.seed(42)
sample_idx = np.random.choice(X_test.index, size=min(200, len(X_test)), replace=False)
X_sample = X_test.loc[sample_idx]

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_sample)
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=True)

In [ ]:
# SHAP global (same for all configs)
shap_global = {}
for cls_idx, cls_name in enumerate(class_names):
    mean_abs = np.abs(shap_values[:, :, cls_idx]).mean(axis=0)
    shap_global[cls_name] = {feat: round(float(val), 6) for feat, val in zip(feature_names, mean_abs)}

shap_global_json = json.dumps(shap_global, indent=2, ensure_ascii=False)
print("SHAP - Importancia Global por Classe:")
print(shap_global_json)

In [ ]:
# SHAP local for N_LOCAL_MAX instances
shap_local_all = []
for idx in range(N_LOCAL_MAX):
    sample_row = X_sample.iloc[idx]
    real_idx = X_sample.index[idx]
    entry = {
        "instance_features": {feat: round(float(sample_row[feat]), 4) for feat in feature_names},
        "true_label": class_names[int(y_test.loc[real_idx])],
        "predicted_label": class_names[int(rf_model.predict(sample_row.to_frame().T)[0])],
        "shap_values_per_class": {}
    }
    for cls_idx, cls_name in enumerate(class_names):
        entry["shap_values_per_class"][cls_name] = {
            feat: round(float(shap_values[idx, f_idx, cls_idx]), 6)
            for f_idx, feat in enumerate(feature_names)
        }
    shap_local_all.append(entry)

print(f"Computed SHAP local for {N_LOCAL_MAX} instances")

In [ ]:
# LIME for N_LOCAL_MAX instances
explainer_l = lime_tabular.LimeTabularExplainer(
    X_train.values,
    feature_names=feature_names,
    class_names=class_names,
    mode='classification'
)

lime_all = []
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="X does not have valid feature names")
    for idx in range(N_LOCAL_MAX):
        sample_row = X_sample.iloc[idx]
        real_idx = X_sample.index[idx]

        exp = explainer_l.explain_instance(
            sample_row.values,
            rf_model.predict_proba,
            num_features=6
        )

        lime_all.append({
            "instance_index": int(real_idx),
            "true_label": class_names[int(y_test.loc[real_idx])],
            "predicted_label": class_names[int(rf_model.predict(sample_row.to_frame().T)[0])],
            "local_prediction_proba": {
                cls: round(float(p), 4)
                for cls, p in zip(class_names, rf_model.predict_proba(sample_row.to_frame().T)[0])
            },
            "feature_contributions": [
                {"rule": rule, "weight": round(float(w), 6)} for rule, w in exp.as_list()
            ]
        })

print(f"Computed LIME for {N_LOCAL_MAX} instances")

# Helpers

In [ ]:
column_description = {
    "Port": "Communication port (encoded: 21=0, 22=1, 23=2, 25=3, 53=4, 80=5, 135=6, 443=7, 4444=8, 6667=9, 8080=10, 31337=11)",
    "Request_Type": "Request type (DNS=0, FTP=1, HTTP=2, HTTPS=3, SMTP=4, SSH=5, Telnet=6)",
    "Protocol": "Transport protocol (ICMP=0, TCP=1, UDP=2)",
    "Payload_Size": "Packet payload size (StandardScaler normalized)",
    "User_Agent": "Client agent (Mozilla/5.0=0, Nikto/2.1.6=1, Wget/1.20.3=2, curl/7.68.0=3, nmap/7.80=4, python-requests/2.25.1=5)",
    "Status": "Request status (Failure=0, Success=1)",
    "Scan_Type_Label": "Target variable: BotAttack=0, Normal=1, PortScan=2"
}

model_info = {
    "type": "Random Forest",
    "task": "Network intrusion detection from log entries",
    "target": "Scan_Type_Label",
    "classes": {"BotAttack": 0, "Normal": 1, "PortScan": 2},
    "features": list(X.columns),
    "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
    "pipeline": "Data cleaning → dropped IP columns → label encoding of categoricals → StandardScaler on Payload_Size → SMOTE class balancing → 70/30 stratified train/test split"
}

system_msg = (
    "You are an expert in Machine Learning, Explainable AI, and Cybersecurity. "
    "You will receive data about a network intrusion detection model that has already been "
    "fully validated through a complete data science pipeline (data cleaning, feature engineering, "
    "label encoding, SMOTE class balancing, and stratified train/test split). "
    "The model achieves high accuracy and is considered reliable. "
    "Your task is NOT to evaluate the model's predictive performance — that is already established. "
    "Your task is to EXPLAIN the model's behavior: why it makes the decisions it does, "
    "which features drive predictions, and what insights a SOC analyst can extract. "
    "After each message, briefly summarize the key data you received (do NOT analyze yet). "
    "When you receive 'OK, start your analysis.', produce your full structured analysis "
    "using ALL data from ALL previous messages."
)


def build_phases(n_samples, n_local):
    """Build Phase 1 and Phase 2 chat chunks for a given config."""
    # Samples
    train_sample = X_train.sample(n_samples, random_state=42)
    train_sample["Scan_Type_Label"] = y_train.loc[train_sample.index]
    train_sample_json = train_sample.to_json(orient="records")

    pred_sample = pd.DataFrame({"real": y_test, "predicted": y_pred})
    pred_sample_json = pred_sample.sample(n_samples, random_state=42).to_json(orient="records")

    # XAI slices
    shap_local = shap_local_all[:n_local]
    lime_explanations = lime_all[:n_local]
    shap_local_json = json.dumps(shap_local, indent=2, ensure_ascii=False)
    lime_explanations_json = json.dumps(lime_explanations, indent=2, ensure_ascii=False)

    # ---- PHASE 1: Raw data only ----
    phase1_msg1 = f"""I will send you data about a network intrusion detection model in parts.
After each part, briefly summarize what you received. Do NOT analyze yet.

# Part 1: Model Info & Column Descriptions

## Model
{json.dumps(model_info, indent=2, ensure_ascii=False)}

## Column Descriptions (all features are numerically encoded)
{json.dumps(column_description, indent=2, ensure_ascii=False)}

**Important:** This model has been through a complete data science pipeline (cleaning, encoding,
balancing, validation). It achieves {model_info['accuracy']*100:.1f}% accuracy on the full test set.
The model is already validated — your goal is to EXPLAIN its behavior, not to evaluate its performance."""

    phase1_msg2 = f"""# Part 2: Representative Data Examples ({n_samples} records)

These are representative examples from the training data to help you understand the feature space
and data distribution. They are NOT the evaluation set.
{train_sample_json}"""

    phase1_msg3 = f"""# Part 3: Sample of Real vs Predicted Labels ({n_samples} records)

These are a small sample from the test set to illustrate prediction patterns.
The model achieves {model_info['accuracy']*100:.1f}% accuracy on the full test set ({len(y_test)} records).
{pred_sample_json}"""

    phase1_task = f"""# OK, start your analysis.

You have received:
- Model info (Random Forest, 3-class intrusion detection, {model_info['accuracy']*100:.1f}% accuracy) + column descriptions
- {n_samples} representative training examples (to understand the feature space)
- {n_samples} sample predictions (the model achieves {model_info['accuracy']*100:.1f}% on the full test set)

The model is already validated and performs well. You do NOT have SHAP or LIME data yet.
Based ONLY on the model info, data patterns, and prediction examples, focus on EXPLAINING
the model's behavior. Provide a structured analysis:

1. **Model Context:** Briefly acknowledge the model's performance (accuracy, per-class balance)
   as context for the explanation that follows. Do not spend effort analyzing accuracy — it is
   already validated. Just establish the baseline so the rest of your analysis has credibility.

2. **Feature Analysis:** Based on the training data, which features seem most important for
   distinguishing BotAttack vs Normal vs PortScan? Rank them and explain from a cybersecurity perspective.

3. **Decision Pattern Hypotheses:** How does the model likely distinguish between attack types?
   What feature combinations drive each class prediction?

4. **Instance-Level Reasoning:** For specific instances, explain what likely drove each prediction.
   What features were decisive?

5. **Cybersecurity Insights:** What patterns in the data are most useful for a SOC analyst?

6. **Model Behavior Assessment:** Based on the data patterns, what are the model's likely decision
   boundaries? Where might it struggle to distinguish between classes?

7. **Improvement Suggestions:** Concrete improvements you would recommend.

Use numbered sections and subsections."""

    # ---- PHASE 2: Add XAI data (same chat) ----
    phase2_msg_shap_global = f"""# NEW DATA: SHAP Global Feature Importance

I now have SHAP (SHapley Additive exPlanations) results for the model.
Mean |SHAP| per feature per class, computed over {len(X_sample)} test samples:
{shap_global_json}"""

    phase2_msg_shap_local = f"""# NEW DATA: SHAP Local Explanations ({n_local} instances)
{shap_local_json}"""

    phase2_msg_lime = f"""# NEW DATA: LIME Local Explanations (same {n_local} instances as SHAP)
{lime_explanations_json}"""

    phase2_task = f"""# OK, now revise your analysis with the XAI evidence.

You previously analyzed the model based only on raw data and prediction examples.
Now you have received SHAP and LIME explanations.

Key SHAP global values (for reference):
{shap_global_json}

Provide a REVISED and UPDATED analysis, focusing on how XAI evidence deepens your
understanding of the model's behavior:

1. **Model Context (Revised):** Has the XAI evidence changed your confidence in the model?
   Briefly note any shifts — then move on to the detailed explanation.

2. **Global Feature Importance (SHAP):** Interpret the importance ranking for each class.
   How does this compare to your earlier feature analysis (without XAI)?
   What did you get right? What did you get wrong?

3. **Local Explanations (SHAP + LIME):** For each of the {n_local} instances, explain the prediction
   using both SHAP values AND LIME rules. Where do they agree? Where do they disagree?

4. **Revised Decision Pattern Analysis:** Do the SHAP/LIME data confirm or contradict your
   earlier hypotheses about how the model distinguishes between classes?

5. **Updated Cybersecurity Insights:** With XAI evidence, what are the strongest indicators
   of BotAttack vs PortScan? How should a SOC analyst use these explanations?

6. **SHAP-LIME Coherence:** Assess agreement between SHAP and LIME.
   Has your understanding of the model's decision boundaries changed?

7. **Updated Improvement Suggestions:** Concrete improvements based on XAI evidence.

Be explicit about what changed in your understanding after seeing the XAI data.
Use numbered sections and subsections."""

    phase1_chunks = [
        {"user": phase1_msg1},
        {"user": phase1_msg2},
        {"user": phase1_msg3},
        {"user": phase1_task, "is_analysis": True},
    ]

    phase2_chunks = [
        {"user": phase2_msg_shap_global},
        {"user": phase2_msg_shap_local},
        {"user": phase2_msg_lime},
        {"user": phase2_task, "is_analysis": True},
    ]

    return phase1_chunks, phase2_chunks

# Execução — Loop over configs

In [ ]:
client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama",
    timeout=httpx.Timeout(REQUEST_TIMEOUT, connect=30.0),
)

all_results = {}   # {(n_samples, n_local): {model_name: {phase1_without_xai, phase2_with_xai}}}
all_timings = {}   # {(n_samples, n_local): {model_name: elapsed}}
previous_model = None

for n_samples, n_local in CONFIGS:
    config_key = f"samples_{n_samples}_local_{n_local}"
    print(f"\n{'#'*60}")
    print(f"# CONFIG: N_SAMPLES={n_samples}, N_LOCAL={n_local}")
    print(f"{'#'*60}")

    phase1_chunks, phase2_chunks = build_phases(n_samples, n_local)

    print("\nPhase 1 (raw data):")
    for i, chunk in enumerate(phase1_chunks, 1):
        print(f"  Msg {i}: {len(chunk['user'])} chars {'[ANALYSIS]' if chunk.get('is_analysis') else ''}")
    print("Phase 2 (+ XAI):")
    for i, chunk in enumerate(phase2_chunks, 1):
        print(f"  Msg {i}: {len(chunk['user'])} chars {'[ANALYSIS]' if chunk.get('is_analysis') else ''}")

    results = {}
    timings = {}

    for model_cfg in models:
        model_name = model_cfg["name"]
        model_size = model_cfg["size"]

        if previous_model:
            !ollama stop $previous_model
            time.sleep(2)

        print(f"\n{'='*60}")
        print(f"Modelo: {model_name} (tier: {model_size}) | N_SAMPLES={n_samples}, N_LOCAL={n_local}")
        print(f"{'='*60}")

        messages = [{"role": "system", "content": system_msg}]
        max_tokens_analysis = 16384 if model_size == "large" else 8192
        model_results = {}

        t_start = time.time()
        try:
            # ---- PHASE 1: Raw data ----
            print(f"\n  --- PHASE 1: Raw data only ---")
            n_p1 = len(phase1_chunks)
            for i, chunk in enumerate(phase1_chunks):
                is_analysis = chunk.get("is_analysis", False)
                messages.append({"role": "user", "content": chunk["user"]})

                if not is_analysis:
                    response = client.chat.completions.create(
                        model=model_name,
                        messages=messages,
                        max_tokens=256
                    )
                    summary = response.choices[0].message.content.strip()
                    messages.append({"role": "assistant", "content": summary})
                    print(f"  P1 Msg {i+1}/{n_p1} sent -> {summary[:80]}")
                else:
                    print(f"  P1 Msg {i+1}/{n_p1} sent -> requesting Phase 1 analysis...")
                    response = client.chat.completions.create(
                        model=model_name,
                        messages=messages,
                        max_tokens=max_tokens_analysis
                    )
                    phase1_result = response.choices[0].message.content
                    messages.append({"role": "assistant", "content": phase1_result})
                    model_results["phase1_without_xai"] = phase1_result
                    print(f"  Phase 1 done: {len(phase1_result)} chars")

            # ---- PHASE 2: Add XAI data (same chat) ----
            print(f"\n  --- PHASE 2: Adding SHAP + LIME ---")
            n_p2 = len(phase2_chunks)
            for i, chunk in enumerate(phase2_chunks):
                is_analysis = chunk.get("is_analysis", False)
                messages.append({"role": "user", "content": chunk["user"]})

                if not is_analysis:
                    response = client.chat.completions.create(
                        model=model_name,
                        messages=messages,
                        max_tokens=256
                    )
                    summary = response.choices[0].message.content.strip()
                    messages.append({"role": "assistant", "content": summary})
                    print(f"  P2 Msg {i+1}/{n_p2} sent -> {summary[:80]}")
                else:
                    print(f"  P2 Msg {i+1}/{n_p2} sent -> requesting Phase 2 analysis...")
                    response = client.chat.completions.create(
                        model=model_name,
                        messages=messages,
                        max_tokens=max_tokens_analysis
                    )
                    phase2_result = response.choices[0].message.content
                    messages.append({"role": "assistant", "content": phase2_result})
                    model_results["phase2_with_xai"] = phase2_result
                    print(f"  Phase 2 done: {len(phase2_result)} chars")

            elapsed = time.time() - t_start
            timings[model_name] = elapsed
            results[model_name] = model_results
            print(f"\n  Total: {elapsed:.1f}s")

        except Exception as e:
            elapsed = time.time() - t_start
            timings[model_name] = elapsed
            model_results["error"] = str(e)
            results[model_name] = model_results
            print(f"  Error: {e} ({elapsed:.1f}s)")

        previous_model = model_name

    all_results[config_key] = results
    all_timings[config_key] = timings

    # Save per config
    out_file = f"resultados_enforce_knowledge_{config_key}.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"\nResultados salvos em {out_file}")

if previous_model:
    !ollama stop $previous_model

print("\n" + "="*60)
print("All runs completed.")
print("="*60)

# Resultados

In [ ]:
for config_key, results in all_results.items():
    print(f"\n{'#'*60}")
    print(f"# CONFIG: {config_key}")
    print(f"{'#'*60}")

    for model_name, model_results in results.items():
        print(f"\n{'='*60}")
        print(f"MODELO: {model_name}")
        print(f"{'='*60}")

        if "phase1_without_xai" in model_results:
            print(f"\n--- PHASE 1: Without XAI ({len(model_results['phase1_without_xai'])} chars) ---\n")
            print(model_results["phase1_without_xai"])

        if "phase2_with_xai" in model_results:
            print(f"\n--- PHASE 2: With XAI ({len(model_results['phase2_with_xai'])} chars) ---\n")
            print(model_results["phase2_with_xai"])

        if "error" in model_results:
            print(f"\nERROR: {model_results['error']}")

        print()

print("\nTimings:")
for config_key, timings in all_timings.items():
    print(f"\n  {config_key}:")
    for name, t in timings.items():
        print(f"    {name}: {t:.1f}s")

In [ ]:
for config_key, results in all_results.items():
    out_file = f"resultados_enforce_knowledge_{config_key}.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"Resultados salvos em {out_file}")